In [7]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.patches import PathPatch
from matplotlib.path import Path
from scipy.spatial import ConvexHull
plt.rcParams.update({
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "font.family": "serif",
    "figure.dpi": 300,
})
import seaborn as sns
import math
import time
import os
import meshio
from network import CNetwork, GNetwork, GNNetwork,FGatedGNN, initialize_weights
from data_sampling import tanh_pattern_sampling, sine_wave_pattern_sampling, random_fixed_sampling
from scipy.interpolate import griddata
from data_sampling import tanh_pattern_sampling, sine_wave_pattern_sampling, random_fixed_sampling
from plot_saver import setup_output_dirs, get_next_run_number

In [8]:
script_name = "i-PINNs-forward"
DIRS = setup_output_dirs(script_name)

In [10]:
class Pinns:
    def __init__(self):

        # Device
        self.device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
        self.network = CNetwork().to(self.device).float()
        print("CUDA Available:", torch.cuda.is_available())

        # Physical parameters
        self.rho = 1.0        # Density [kg/m³]
        self.eta_0 = 0.01     # Zero-shear viscosity [Pa·s]
        self.eta_inf = 0.005  # Infinite-shear viscosity [Pa·s]
        self.lam = 1.0         # Relaxation time [s]
        self.n_cy = 0.5       # Power-law index
        self.a_cy = 2.0      # Yasuda parameter

        #Read mesh
        mesh = meshio.read(r"Input/mesh.msh")
        coords_np = mesh.points[:, :2]
        cells      = mesh.cells
        
        #Extract boundary node indices
        wall1_edges  = cells[1].data
        wall2_edges  = cells[2].data
        inlet_edges  = cells[3].data
        outlet_edges = cells[4].data
        
        def edge_nodes(edges, coords):
            idx = np.unique(edges.flatten())
            return coords[idx]
        
        uw_coords     = edge_nodes(wall1_edges,  coords_np)   # upper wall
        dw_coords     = edge_nodes(wall2_edges,  coords_np)   # lower wall
        inlet_coords  = edge_nodes(inlet_edges,  coords_np)   # inlet
        outlet_coords = edge_nodes(outlet_edges, coords_np)   # outlet
        
        # Sort nodes along their natural direction
        uw_coords     = uw_coords    [np.argsort(uw_coords    [:, 0])]
        dw_coords     = dw_coords    [np.argsort(dw_coords    [:, 0])]
        inlet_coords  = inlet_coords [np.argsort(inlet_coords [:, 1])]
        outlet_coords = outlet_coords[np.argsort(outlet_coords[:, 1])]
        
        #Geometry parameters derived from the mesh
        x_min  = coords_np[:, 0].min()
        x_max  = coords_np[:, 0].max()
        y_min  = coords_np[:, 1].min()
        y_max  = coords_np[:, 1].max()
        self.L = float(x_max - x_min)
        self.H = float(y_max - y_min)
        
        #Interior collocation points (from mesh interior nodes)
        boundary_idx = np.unique(np.concatenate([
            wall1_edges.flatten(),
            wall2_edges.flatten(),
            inlet_edges.flatten(),
            outlet_edges.flatten(),
        ]))
        interior_mask = np.ones(coords_np.shape[0], dtype=bool)
        interior_mask[boundary_idx] = False
        interior_coords = coords_np[interior_mask]
        
        self.X = torch.tensor(interior_coords, dtype=torch.float32)\
                       .to(self.device).requires_grad_(True)
        
        #Convert boundary arrays to tensors
        to_t = lambda arr: torch.tensor(arr, dtype=torch.float32)
        
        inlet_t  = to_t(inlet_coords)
        uw_t     = to_t(uw_coords)
        dw_t     = to_t(dw_coords)
        outlet_t = to_t(outlet_coords)

        
        # X_train  (all velocity BC points)
        self.inlet = inlet_t
        self.uw    = uw_t
        self.dw    = dw_t
        self.X_train = torch.cat([inlet_t, uw_t, dw_t], dim=0).to(self.device)
        
        #X_train_p (pressure BC – outlet)
        self.X_train_p  = outlet_t.to(self.device)
        
        #Boundary condition values
        y1 = inlet_coords[:, 1]
        
        u_in = torch.tensor(1.5 * (1.0 - 4*y1**2), dtype=torch.float32)
        v_in = torch.zeros_like(u_in)
        
        self.uv_inlet = torch.stack((u_in, v_in), dim=1)
        
        # upper wall no-slip
        n_uw = uw_coords.shape[0]
        self.uv_uw = torch.zeros(n_uw, 2)

        # lower wall no-slip
        n_dw = dw_coords.shape[0]
        self.uv_dw = torch.zeros(n_dw, 2)
        
        self.uv_train = torch.cat(
            [self.uv_inlet, self.uv_uw, self.uv_dw],
            dim=0
        ).to(self.device)

        # outlet: p = 0
        self.p_train = torch.zeros(outlet_t.shape[0], device=self.device)

        # Loss
        self.criterion = nn.MSELoss()

        # Optimizers
        self.adam = torch.optim.Adam(self.network.parameters(), lr=1e-4)

        self.optimizer = torch.optim.LBFGS(
            self.network.parameters(),
            lr=0.01,
            max_iter=2000,
            max_eval=2000,
            history_size=50,
            tolerance_grad=1e-8,
            tolerance_change=1.0 * np.finfo(float).eps,
            line_search_fn="strong_wolfe"
        )

        self.loss_history = []

    # gradient function
    def gradient(self, y, x, index):
        grad = torch.autograd.grad(
            y, x,
            grad_outputs=torch.ones_like(y),
            create_graph=True,
            retain_graph=True,
            only_inputs=True
        )[0]
        return grad[:, index]


    # Loss function
    def loss_f(self):
        # Boundary 
        uv_b = self.network(self.X_train)
        u_b = uv_b[:, 0]
        v_b = uv_b[:, 1]
        
        p_b = self.network(self.X_train_p)[:, 2]
        
        loss_u_b = self.criterion(u_b, self.uv_train[:, 0])
        loss_v_b = self.criterion(v_b, self.uv_train[:, 1])
        loss_p_b = self.criterion(p_b, self.p_train)
        loss_data = loss_u_b + loss_v_b +loss_p_b
    
        # Interior
        uvp = self.network(self.X)
        self.u_P = uvp[:, 0]
        self.v_P = uvp[:, 1]
        self.P_P = uvp[:, 2]
    
        # derivatives
        self.du_dx = self.gradient(self.u_P, self.X, 0)
        self.du_dy = self.gradient(self.u_P, self.X, 1)
        self.du_dxx = self.gradient(self.du_dx, self.X, 0)
        self.du_dyy = self.gradient(self.du_dy, self.X, 1)
    
        self.dv_dx = self.gradient(self.v_P, self.X, 0)
        self.dv_dy = self.gradient(self.v_P, self.X, 1)
        self.dv_dxx = self.gradient(self.dv_dx, self.X, 0)
        self.dv_dyy = self.gradient(self.dv_dy, self.X, 1)
    
        self.dp_dx = self.gradient(self.P_P, self.X, 0)
        self.dp_dy = self.gradient(self.P_P, self.X, 1)
    
        # Navier-Stokes residuals
        gamma_dot = torch.sqrt(2*self.du_dx**2 + 2*self.dv_dy**2 + (self.du_dy + self.dv_dx)**2 + 1e-12)
        mu = self.eta_inf + (self.eta_0 - self.eta_inf) * \
             (1 + (self.lam * gamma_dot)**self.a_cy)**((self.n_cy - 1)/self.a_cy)
        # stress components
        tau_xx = 2*mu*self.du_dx
        tau_xy = mu*(self.du_dy + self.dv_dx)
        tau_yy = 2*mu*self.dv_dy
        
        # divergence of stress
        dtau_xx_dx = self.gradient(tau_xx, self.X, 0)
        dtau_xy_dy = self.gradient(tau_xy, self.X, 1)
        dtau_xy_dx = self.gradient(tau_xy, self.X, 0)
        dtau_yy_dy = self.gradient(tau_yy, self.X, 1)

        # Navier-Stokes equations residuals
        continuity = self.du_dx + self.dv_dy
        momentum_x = self.rho * (self.u_P * self.du_dx + self.v_P * self.du_dy) \
             + self.dp_dx \
             - (dtau_xx_dx + dtau_xy_dy)
        momentum_y = self.rho * (self.u_P * self.dv_dx + self.v_P * self.dv_dy) \
             + self.dp_dy \
             - (dtau_xy_dx + dtau_yy_dy)
    
        loss_continuity = self.criterion(continuity, torch.zeros_like(continuity))
        loss_momentum_x = self.criterion(momentum_x, torch.zeros_like(momentum_x))
        loss_momentum_y = self.criterion(momentum_y, torch.zeros_like(momentum_y))
    
        loss_pde = loss_continuity + loss_momentum_x + loss_momentum_y
        loss_total = loss_data + loss_pde
    
        return loss_total, loss_data, loss_continuity, loss_momentum_x, loss_momentum_y
    
    # Training loop
    def train(self, num_epochs=5000):
        self.network.train()
        run_number    = get_next_run_number(script_name)
        model_path    = os.path.join(DIRS["models"],  "model_best.pth")
        history_path  = os.path.join(DIRS["history"], f"loss_history_run{run_number:03d}.csv")
        time_path = os.path.join(DIRS["history"], f"adam_epoch_time_run{run_number:03d}.csv")
        print(f"Run {run_number:03d}")
        print(f"   Model   -> {model_path}  (overwrites previous)")
        print(f"   History -> {history_path}")
        epoch_times = []
        checkpoint_path = "pinns_checkpoint.pth"
    
        for i in range(num_epochs):
            start_time = time.time()
            self.adam.zero_grad()
            loss_total, loss_data, loss_continuity, loss_momentum_x, loss_momentum_y = self.loss_f()
            loss_total.backward()
            self.adam.step()
    
            self.loss_history.append({"epoch": i, "loss_total": loss_total.item()})
            epoch_duration = time.time() - start_time
            epoch_times.append({"epoch": i, "duration": epoch_duration})    
            if i % 10 == 0:
                print(f"Epoch {i}: Total Loss={loss_total.item():.3e}, "
                      f"Boundary Loss={loss_data.item():.3e}, "
                      f"Continuity Loss={loss_continuity.item():.3e}, "
                      f"Momentum X Loss={loss_momentum_x.item():.3e}, "
                      f"Momentum Y Loss={loss_momentum_y.item():.3e}")
            if i % 100 == 0:
                torch.save(self.network.state_dict(), model_path)
                print(f"  Checkpoint saved -> {model_path}")
                
        df_time = pd.DataFrame(epoch_times)
        df_time.to_csv(time_path, index=False)
        print(f"Adam epoch times saved -> {time_path}")
                
        # LBFGS phase
        lbfgs_step = [0]
        adam_epochs  = num_epochs
        def closure():
            self.optimizer.zero_grad()
            loss_total, loss_data, loss_continuity, loss_momentum_x, loss_momentum_y = self.loss_f()
            loss_total.backward()
            lbfgs_step[0] += 1

            self.loss_history.append({
                "epoch": adam_epochs + lbfgs_step[0],
                "loss_total": loss_total.item()
            })
            
            print(f"[LBFGS step {lbfgs_step[0]}] Loss: Total={loss_total.item():.3e}, "
                  f"Boundary={loss_data.item():.3e}, "
                  f"Continuity={loss_continuity.item():.3e}, "
                  f"Momentum X={loss_momentum_x.item():.3e}, "
                  f"Momentum Y={loss_momentum_y.item():.3e}")
            return loss_total
        self.optimizer.step(closure)
        print("Starting LBFGS...")
        torch.save(self.network.state_dict(), model_path)
        print(f"Final model saved -> {model_path}")
        df_run = pd.DataFrame(self.loss_history)
        df_run.to_csv(history_path, index=False)
        print(f"Run history saved -> {history_path}")

    def plot(self):
        self.network.eval()
        
        # Load mesh
        mesh = meshio.read(r"Input/mesh.msh")
        coords_np = mesh.points[:, :2]
        x_mesh, y_mesh = coords_np[:, 0], coords_np[:, 1]
        xy_mesh = torch.tensor(coords_np, dtype=torch.float32).to(self.device)
        
        # Extract boundary segments
        boundary_segments = np.vstack([mesh.cells[i].data for i in [1, 2, 3, 4]])
        
        # PINN predictions on mesh
        with torch.no_grad():
            uvp_pred = self.network(xy_mesh)
            u_pred = uvp_pred[:, 0].cpu().numpy()
            v_pred = uvp_pred[:, 1].cpu().numpy()
            p_pred = uvp_pred[:, 2].cpu().numpy()
        
        # Viscosity on mesh
        xy_mesh_g = xy_mesh.clone().detach().requires_grad_(True)
        uvp = self.network(xy_mesh_g)
        u, v = uvp[:, 0], uvp[:, 1]
        
        du_dx = self.gradient(u, xy_mesh_g, 0)
        du_dy = self.gradient(u, xy_mesh_g, 1)
        dv_dx = self.gradient(v, xy_mesh_g, 0)
        dv_dy = self.gradient(v, xy_mesh_g, 1)
        
        gamma_dot = torch.sqrt(2*du_dx**2 + 2*dv_dy**2 + (du_dy + dv_dx)**2 + 1e-12)
        eta_pinn = self.eta_inf + (self.eta_0 - self.eta_inf) * (1 + (self.lam * gamma_dot)**self.a_cy)**((self.n_cy - 1)/self.a_cy)
        eta_pred = eta_pinn.detach().cpu().numpy()
        
        # Load CFD data
        df = pd.read_csv("Input/data")
        df.columns = df.columns.str.strip()
        x_cfd = df["x-coordinate"].values
        y_cfd = df["y-coordinate"].values
        u_cfd = df["x-velocity"].values
        v_cfd = df["y-velocity"].values
        p_cfd = df["pressure"].values
        viscosity_cfd = df["viscosity-lam"].values
        
        # PINN at CFD points
        xy_cfd_torch = torch.from_numpy(np.column_stack((x_cfd, y_cfd))).float().to(self.device)
        with torch.no_grad():
            uvp_pinn_cfd = self.network(xy_cfd_torch)
            u_pinn_cfd = uvp_pinn_cfd[:, 0].cpu().numpy()
            v_pinn_cfd = uvp_pinn_cfd[:, 1].cpu().numpy()
            p_pinn_cfd = uvp_pinn_cfd[:, 2].cpu().numpy()
        
        # Compute errors
        eps = 1e-12
        u_range = u_cfd.max() - u_cfd.min() + eps
        v_range = np.abs(v_cfd).max() + eps
        p_range = p_cfd.max() - p_cfd.min() + eps
        
        p_pinn_shifted = p_pinn_cfd - p_pinn_cfd.min()
        p_cfd_shifted = p_cfd - p_cfd.min()
        
        rel_err_u = 100.0 * np.abs(u_pinn_cfd - u_cfd) / u_range
        rel_err_v = 100.0 * np.abs(v_pinn_cfd - v_cfd) / v_range
        rel_err_p = 100.0 * np.abs(p_pinn_shifted - p_cfd_shifted) / p_range
        
        def compute_l2(pred, ref):
            return 100.0 * np.linalg.norm(pred - ref) / (np.linalg.norm(ref) + eps)
        
        l2_u = compute_l2(u_pinn_cfd, u_cfd)
        l2_v = compute_l2(v_pinn_cfd, v_cfd)
        l2_p = compute_l2(p_pinn_shifted, p_cfd_shifted)
        
        # Sort boundary for masking
        def sort_boundary(segments, points):
            adjacency = {}
            for a, b in segments:
                adjacency.setdefault(a, []).append(b)
                adjacency.setdefault(b, []).append(a)
            
            start = segments[0][0]
            path = [start]
            prev, current = None, start
            
            while True:
                neighbors = adjacency[current]
                next_node = next((n for n in neighbors if n != prev), None)
                if next_node is None or next_node == start:
                    break
                path.append(next_node)
                prev, current = current, next_node
            return path
        
        # Plot with boundary mask
        def tricontour_plot(x, y, values, title, fname, label, cmap="jet"):
            fpath = os.path.join(DIRS["plots"], fname)
            fig, ax = plt.subplots(figsize=(12, 3))
            cf = ax.tricontourf(x, y, values, levels=80, cmap=cmap)
            plt.colorbar(cf, label=label, ax=ax)
            
            ax.set_xlim(0, 10)
            ax.set_ylim(-0.6, 0.6)
            
            # Create mask
            boundary_points = sort_boundary(boundary_segments, mesh.points)
            xy = np.array([[mesh.points[i][0], mesh.points[i][1]] for i in boundary_points])
            
            x_min, x_max = x.min() - 1, x.max() + 1
            y_min, y_max = y.min() - 1, y.max() + 1
            outer_box = np.array([[x_min, y_min], [x_max, y_min], 
                                  [x_max, y_max], [x_min, y_max], [x_min, y_min]])
            
            vertices = np.vstack([outer_box, xy[::-1]])
            codes = ([Path.MOVETO] + [Path.LINETO] * (len(outer_box) - 2) + [Path.CLOSEPOLY] +
                     [Path.MOVETO] + [Path.LINETO] * (len(xy) - 2) + [Path.CLOSEPOLY])
            
            clip_path = PathPatch(Path(vertices, codes), facecolor='white',edgecolor='none', zorder=2)
            ax.add_patch(clip_path)
            
            # Draw boundary
            for seg in boundary_segments:
                p1, p2 = mesh.points[seg[0]], mesh.points[seg[1]]
                ax.plot([p1[0], p2[0]], [p1[1], p2[1]], 'k-', linewidth=1, zorder=3)
            
            ax.set_xlabel("x [m]")
            ax.set_ylabel("y [m]")
            ax.set_title(title)
            plt.tight_layout()
            plt.savefig(fpath, dpi=300, bbox_inches='tight')
            plt.show()
            plt.close()
        
        # PINN results
        tricontour_plot(x_mesh, y_mesh, u_pred, "PINN: x-velocity (u)", "u_pinn.png", "u [m/s]")
        tricontour_plot(x_mesh, y_mesh, v_pred, "PINN: y-velocity (v)", "v_pinn.png", "v [m/s]")
        tricontour_plot(x_mesh, y_mesh, p_pred, "PINN: Pressure (p)", "p_pinn.png", "p [Pa]")
        tricontour_plot(x_mesh, y_mesh, eta_pred, "PINN: Viscosity (η)", "viscosity_pinn.png", "η [Pa·s]")
        
        # CFD results
        tricontour_plot(x_cfd, y_cfd, u_cfd, "CFD: x-velocity (u)", "u_cfd.png", "u [m/s]")
        tricontour_plot(x_cfd, y_cfd, v_cfd, "CFD: y-velocity (v)", "v_cfd.png", "v [m/s]")
        tricontour_plot(x_cfd, y_cfd, p_cfd, "CFD: Pressure (p)", "p_cfd.png", "p [Pa]")
        tricontour_plot(x_cfd, y_cfd, viscosity_cfd, "CFD: Viscosity (η)", "viscosity_cfd.png", "η [Pa·s]")
        
        # Error plots
        tricontour_plot(x_cfd, y_cfd, rel_err_u, f"Relative Error u (%)\nL2={l2_u:.2f}%", 
                        "error_u_rel.png", "Error [%]", "viridis")
        tricontour_plot(x_cfd, y_cfd, rel_err_v, f"Relative Error v (%)\nL2={l2_v:.2f}%", 
                        "error_v_rel.png", "Error [%]", "viridis")
        tricontour_plot(x_cfd, y_cfd, rel_err_p, f"Relative Error p (%)\nL2={l2_p:.2f}%", 
                        "error_p_rel.png", "Error [%]", "viridis")
        
        # Inlet profile
        ny_prof = 200
        y_prof = torch.linspace(-self.H/2, self.H/2, ny_prof, device=self.device).float()
        x_prof = torch.zeros_like(y_prof)
        xy_inlet = torch.stack([x_prof, y_prof], dim=1)
        
        with torch.no_grad():
            u_pinn_in = self.network(xy_inlet)[:, 0].cpu().numpy()
        
        y_np = y_prof.cpu().numpy()
        u_target = 1.5 * (1.0 - 4.0 * y_np**2)
        
        fpath = os.path.join(DIRS["plots"], "u_profile_inlet.png")
        plt.figure(figsize=(4, 6))
        plt.plot(u_pinn_in, y_np, 'r-', lw=2, label='PINN')
        plt.plot(u_target, y_np, 'k--', lw=2, label='Analytical')
        plt.xlabel('u [m/s]')
        plt.ylabel('y [m]')
        plt.title('Velocity profile at inlet')
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(fpath, dpi=300, bbox_inches='tight')
        plt.show()
        plt.close()

In [ ]:
net = Pinns()
net.network.load_state_dict(torch.load(os.path.join(DIRS["models"], "model_best.pth")))
net.train(num_epochs=100000)
net.plot()

CUDA Available: True


Warning: Unknown index 4. Skipping.

Warning: Zone specification not supported yet (interior, interior-surface_body). Skipping.

Warning: Zone specification not supported yet (fluid, surface_body). Skipping.

Warning: Zone specification not supported yet (wall, wall1). Skipping.

...

Warning: Zone specification not supported yet (velocity-inlet, inlet). Skipping.

Warning: Zone specification not supported yet (pressure-outlet, outlet). Skipping.

Run 003
   Model   -> i-PINNs-forward-Results/models/model_best.pth  (overwrites previous)
   History -> i-PINNs-forward-Results/history/loss_history_run003.csv
Epoch 0: Total Loss=3.246e-03, Boundary Loss=2.496e-03, Continuity Loss=6.544e-04, Momentum X Loss=9.039e-05, Momentum Y Loss=5.298e-06
  Checkpoint saved -> i-PINNs-forward-Results/models/model_best.pth


/tmp/ipykernel_493536/2879254546.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  net.network.load_state_dict(torch.load(os.path.join(DIRS["models"], "model_best.pth")))


Epoch 10: Total Loss=3.519e-03, Boundary Loss=2.719e-03, Continuity Loss=6.096e-04, Momentum X Loss=1.822e-04, Momentum Y Loss=8.973e-06
Epoch 20: Total Loss=3.322e-03, Boundary Loss=2.321e-03, Continuity Loss=8.875e-04, Momentum X Loss=1.067e-04, Momentum Y Loss=7.178e-06
Epoch 30: Total Loss=3.260e-03, Boundary Loss=2.409e-03, Continuity Loss=7.592e-04, Momentum X Loss=8.697e-05, Momentum Y Loss=5.496e-06
Epoch 40: Total Loss=3.250e-03, Boundary Loss=2.544e-03, Continuity Loss=6.051e-04, Momentum X Loss=9.509e-05, Momentum Y Loss=5.787e-06
Epoch 50: Total Loss=3.248e-03, Boundary Loss=2.528e-03, Continuity Loss=6.215e-04, Momentum X Loss=9.293e-05, Momentum Y Loss=5.522e-06
Epoch 60: Total Loss=3.246e-03, Boundary Loss=2.493e-03, Continuity Loss=6.581e-04, Momentum X Loss=9.024e-05, Momentum Y Loss=5.356e-06
Epoch 70: Total Loss=3.246e-03, Boundary Loss=2.483e-03, Continuity Loss=6.682e-04, Momentum X Loss=8.959e-05, Momentum Y Loss=5.327e-06
Epoch 80: Total Loss=3.246e-03, Boundary 

In [2]:
import shutil
import os
folder_path = 'i-PINNs-forward-Results'
#shutil.rmtree(folder_path)